# 03 Ranking Evaluation

This notebook explains what happens after candidate generation.

The demo reranker is intentionally simple and transparent:
- dot product between user interests and campaign weights,
- plus bid,
- plus a small freshness term,
- minus a lightweight frequency penalty.

That makes the ranking behavior easy to inspect while still being realistic enough for offline evaluation.

In [ ]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.candidate import build_indexes, filter_campaigns_for_user, generate_candidates_in_memory
from app.models import Campaign, UserProfile
from app.ranking import rerank_campaigns, score_campaign
from data.common import click_probability, read_jsonl, truth_score
from experiments.evaluate import evaluate_synthetic

DATASET_DIR = REPO_ROOT / 'data' / 'generated' / 'synthetic'
users = [UserProfile.model_validate(row) for row in read_jsonl(DATASET_DIR / 'users.jsonl')]
campaigns = [Campaign.model_validate(row) for row in read_jsonl(DATASET_DIR / 'campaigns.jsonl')]
campaign_by_id = {campaign.campaign_id: campaign for campaign in campaigns}
indexes = build_indexes(campaigns)

sample_user = next(user for user in users if generate_candidates_in_memory(user, indexes, max_candidates=50, strong_signal_count=2))
sample_user

## From Candidate IDs To Ranked Ads

The notebook below follows the same steps as the API:
1. generate candidate IDs,
2. materialize campaign objects,
3. enforce eligibility,
4. score and sort the survivors.

In [ ]:
candidate_ids = generate_candidates_in_memory(sample_user, indexes, max_candidates=50, strong_signal_count=2)
candidate_campaigns = [campaign_by_id[campaign_id] for campaign_id in candidate_ids if campaign_id in campaign_by_id]
eligible_campaigns = filter_campaigns_for_user(sample_user, candidate_campaigns)
ranked = rerank_campaigns(sample_user, eligible_campaigns, top_k=10)

pd.Series(
    {
        'candidate_ids': len(candidate_ids),
        'eligible_campaigns_after_filter': len(eligible_campaigns),
        'top_k_returned': len(ranked),
    }
)

## Score Components

The returned score is decomposed into named parts so the ranking can be explained to a teammate or customer.
That same component view is what the `/rank` endpoint returns in the demo.

In [ ]:
score_frame = pd.DataFrame(
    [
        {
            'campaign_id': item.campaign_id,
            'score': item.score,
            **item.score_components,
            'click_probability': round(click_probability(sample_user, campaign_by_id[item.campaign_id]), 6),
            'truth_score': round(truth_score(sample_user, campaign_by_id[item.campaign_id]), 6),
        }
        for item in ranked
    ]
)
score_frame

## Single-Campaign Explanation

For one campaign, it is useful to inspect both the raw campaign object and the scored breakdown.

In [ ]:
best = ranked[0]
best_campaign = campaign_by_id[best.campaign_id]

display(pd.json_normalize(best_campaign.model_dump()))
display(pd.Series(score_campaign(sample_user, best_campaign).model_dump()))

## Offline Quality Metrics

The synthetic evaluator reports both ranking quality and candidate-generation recall.
That separation matters because a perfect reranker cannot recover a campaign that was dropped during coarse retrieval.

In [ ]:
results = evaluate_synthetic(DATASET_DIR, top_k=5, sample_users=250)
pd.Series(results)